# Unpaired two-sample t-test

**This Jupyter notebook does the following:**

- Uses used car data to compare the prices of two car makes (grouped boxplot)
- Performs an unpaired two-sample t-test (Welch's t-test, incl. Levene test for equal variances) and interprets the result
- Applies the t-test to the apartment data to compare the `price_per_m2` between densely populated and other municipalities

## Libraries and settings

In [ ]:
# Libraries
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats

# Settings for seaborn
sns.set_theme(style="ticks", palette="pastel")

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Show current working directory
print(os.getcwd())

## Read example data (used car data)

In [ ]:
df = pd.read_excel('./Data/t-tests_and_ANOVA.xlsx', sheet_name='t-test_data')

# Show categories of 'Make'
print(df['Make'].value_counts())

# Show first rows in dataframe
df.head()

## Create grouped boxplot (groups = make, values = car prices)

In [ ]:
# Plot boxplot with groups
plt.figure(figsize=(8,2))
ax = sns.boxplot(x="Price",
                 y="Make",
                 hue="Make",
                 palette=['r', 'g'],
                 legend=False,
                 data=df,
                 orient='h')
plt.grid()
plt.show()

## Unpaired two-sample t-test

In [ ]:
# Create subsets (groups)
vw = df.loc[df['Make'] == 'VW']
bmw = df.loc[df['Make'] == 'BMW']

# Check whether the variances of both groups are equal (Levene test, H0: equal variances)
print(f"Levene test p-value: {stats.levene(vw['Price'], bmw['Price']).pvalue:.4f}")

# Create t-test (Welch's t-test, i.e. equal variances are not assumed)
result = stats.ttest_ind(vw['Price'], bmw['Price'], equal_var=False)

# Print result
print(f'Test-statistic: {result.statistic:.5f}, p-value: {result.pvalue:.4g}')

## Interpretation of result

<p> Because the p-value is lower than 0.05, the null hypothesis can be rejected, i.e. there is evidence that the used car prices of VW and BMW differ (5% significance level). The negative test statistic shows that the mean price of the first group (VW) is lower than the mean price of the second group (BMW).</p>
<p> Note: Welch's t-test was used because it does not require equal variances in both groups. If the variances are equal, it gives almost the same result as the classical (Student's) t-test.</p>

## Importing apartment data

In [ ]:
# Read the data to a pandas data frame
df = pd.read_csv('./Data/apartments_data_enriched_cleaned.csv', 
                 sep=';', 
                 encoding='utf-8')[['web-scraper-order',
                                    'address_raw',
                                    'lat',
                                    'lon',
                                    'bfs_number',
                                    'bfs_name',
                                    'rooms', 
                                    'area', 
                                    'luxurious', 
                                    'price', 
                                    'price_per_m2',
                                    'pop_dens',
                                    'frg_pct',
                                    'mean_taxable_income',
                                    'dist_supermarket']]

# Remove duplicated listings (same address, rooms, area and price; the
# 'web-scraper-order' differs for every scraped row and is therefore not used)
df = df.drop_duplicates(subset=['address_raw', 'rooms', 'area', 'price'])

# Get number of rows and columns
print(df.shape)

# Show first records
df.head(5)

## Create new binary variable with densely populated and all other municipalities

In [ ]:
df['pop_dens_binary'] = (df['pop_dens'] >= 1000).astype(int)
df.head(5)

## Create pivot table with mean price_per_m2

In [ ]:
# Using pivot_table to reshape the data and calculate means 
pd.pivot_table(df[['pop_dens_binary', 'price_per_m2']],
               index=['pop_dens_binary'],
               values=['price_per_m2'],
               aggfunc=[np.mean, 'count'])

## Create grouped boxplot (groups = pop_dens_binary, values = prices_per_m2)

In [ ]:
# Plot boxplot with groups
plt.figure(figsize=(8,2))
ax = sns.boxplot(x="price_per_m2",
                 y="pop_dens_binary",
                 hue="pop_dens_binary",
                 data=df,
                 palette=['r', 'g'],
                 legend=False,
                 orient='h')
plt.grid()
plt.show()

## Unpaired two-sample t-test

In [ ]:
# Create subsets (groups)
densely_populated = df.loc[df['pop_dens_binary'] == 1]
not_densely_populated = df.loc[df['pop_dens_binary'] == 0]

# Check whether the variances of both groups are equal (Levene test, H0: equal variances)
print(f"Levene test p-value: {stats.levene(densely_populated['price_per_m2'], not_densely_populated['price_per_m2']).pvalue:.4g}")

# Create t-test (Welch's t-test, i.e. equal variances are not assumed)
result = stats.ttest_ind(densely_populated['price_per_m2'],
                         not_densely_populated['price_per_m2'],
                         equal_var=False)

# Print result
print(f'Test-statistic: {result.statistic:.5f}, p-value: {result.pvalue:.4g}')

## Interpretation of result

<p> Because the p-value is lower than 0.05, the null hypothesis can be rejected, i.e. there is evidence that the prices per m2 differ between densely populated and other municipalities (5% significance level). The positive test statistic (and the pivot table above) shows that the prices per m2 are higher in densely populated municipalities.</p>
<p> Note: The Levene test shows that the variances of the two groups differ. Therefore, Welch's t-test was used, which does not assume equal variances.</p>

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')